# Practical Session 03
## Part I - Instructor-led mini-lab
### Example 1 - Data arrays, scaling, and a baseline

We start from a small supervised regression problem. The goal is not to use a sophisticated model, but to keep the numerical structure visible.

Before running the code, think about the following questions:

1. Which axis labels samples?
2. Which axis labels features?
3. Which quantities are allowed to be computed from the training set only?
4. What is a useful baseline before fitting any model?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
rng = np.random.default_rng(2016)

N = 600

time = rng.uniform(0.0, 10.0, size=N)
field = rng.uniform(-1.0, 1.0, size=N)
temperature = 300.0 + 5.0 * rng.normal(size=N)

X = np.column_stack([time, field, temperature])

theta_true = np.array([0.45, -1.20, 0.08])
bias_true = -23.0
noise = 0.25 * rng.normal(size=N)

y = X @ theta_true + bias_true + noise

print("X.shape:", X.shape)
print("y.shape:", y.shape)
print("feature means:", X.mean(axis=0))
print("feature scales:", X.std(axis=0))

assert X.ndim == 2
assert y.ndim == 1
assert X.shape[0] == y.shape[0]


The features have different numerical scales. The optimizer will see these numbers directly, so preprocessing is part of the numerical experiment.

We now split the data and fit the standardization only on the training subset.


In [ ]:
idx = rng.permutation(N)

n_train = int(0.70 * N)
n_val = int(0.15 * N)

train_idx = idx[:n_train]
val_idx = idx[n_train:n_train + n_val]
test_idx = idx[n_train + n_val:]

X_train, y_train = X[train_idx], y[train_idx]
X_val, y_val = X[val_idx], y[val_idx]
X_test, y_test = X[test_idx], y[test_idx]

mu = X_train.mean(axis=0)
scale = X_train.std(axis=0)
scale[scale == 0.0] = 1.0

X_train_s = (X_train - mu) / scale
X_val_s = (X_val - mu) / scale
X_test_s = (X_test - mu) / scale

print("X_train_s.shape:", X_train_s.shape)
print("X_val_s.shape:  ", X_val_s.shape)
print("X_test_s.shape: ", X_test_s.shape)
print("training mean after scaling:", X_train_s.mean(axis=0))
print("training std after scaling: ", X_train_s.std(axis=0))

assert X_train_s.shape == X_train.shape
assert X_val_s.shape == X_val.shape
assert X_test_s.shape == X_test.shape


In [ ]:
def mse_loss(y_pred, y):
    residual = y_pred - y
    return 0.5 * np.mean(residual**2)

baseline = y_train.mean()

train_baseline_loss = mse_loss(np.full_like(y_train, baseline), y_train)
val_baseline_loss = mse_loss(np.full_like(y_val, baseline), y_val)

print(f"training baseline loss:   {train_baseline_loss:.6f}")
print(f"validation baseline loss: {val_baseline_loss:.6f}")

Questions:

1. Why is the validation set not used when computing `mu` and `scale`?
2. Why is the mean predictor a useful baseline?
3. Which objects would you save if you wanted to reproduce this numerical experiment later?


### Example 2 - A minimal gradient-descent training loop

For a linear model

$$
\hat{y}=X\theta+b,
$$

the residual and mean squared loss are

$$
r=X\theta+b-y,
\qquad
L(\theta,b)=\frac{1}{2N}\sum_{i=1}^N r_i^2.
$$

The analytical gradients are

$$
\nabla_\theta L=\frac{1}{N}X^T r,
\qquad
\frac{\partial L}{\partial b}=\frac{1}{N}\sum_{i=1}^N r_i.
$$


In [ ]:
def linear_predict(X, theta, bias):
    return X @ theta + bias


def linear_gradients(X, y, theta, bias):
    y_pred = linear_predict(X, theta, bias)
    residual = y_pred - y

    grad_theta = X.T @ residual / residual.size
    grad_bias = residual.mean()

    return grad_theta, grad_bias


def train_linear_regression(X_train, y_train, X_val, y_val, eta=0.05, n_steps=500):
    theta = np.zeros(X_train.shape[1])
    bias = 0.0

    train_history = []
    val_history = []
    grad_norm_history = []

    for step in range(n_steps):
        y_train_pred = linear_predict(X_train, theta, bias)
        train_loss = mse_loss(y_train_pred, y_train)

        grad_theta, grad_bias = linear_gradients(X_train, y_train, theta, bias)
        grad_norm = np.sqrt(np.sum(grad_theta**2) + grad_bias**2)

        theta -= eta * grad_theta
        bias -= eta * grad_bias

        y_val_pred = linear_predict(X_val, theta, bias)
        val_loss = mse_loss(y_val_pred, y_val)

        train_history.append(train_loss)
        val_history.append(val_loss)
        grad_norm_history.append(grad_norm)

        if not np.isfinite(train_loss) or not np.isfinite(val_loss):
            break

    return {
        "theta": theta,
        "bias": bias,
        "train_history": np.array(train_history),
        "val_history": np.array(val_history),
        "grad_norm_history": np.array(grad_norm_history),
    }


In [ ]:
result = train_linear_regression(
    X_train_s,
    y_train,
    X_val_s,
    y_val,
    eta=0.05,
    n_steps=600,
)

print("theta in scaled coordinates:", result["theta"])
print("bias:", result["bias"])
print(f"final training loss:   {result['train_history'][-1]:.6f}")
print(f"final validation loss: {result['val_history'][-1]:.6f}")
print(f"final gradient norm:   {result['grad_norm_history'][-1]:.3e}")
print("finite training history:", np.isfinite(result["train_history"]).all())


In [ ]:
fig, ax = plt.subplots(figsize=(6.0, 3.3))

ax.plot(result["train_history"], label="training")
ax.plot(result["val_history"], label="validation")

ax.set_xlabel("step")
ax.set_ylabel("loss")
ax.set_yscale("log")
ax.set_title("Gradient descent diagnostics")
ax.grid(True, alpha=0.3)
ax.legend()

fig.tight_layout()


Questions:

1. Which part of the code is the forward computation?
2. Which arrays have the same shape as the model parameters?
3. Why do we monitor validation loss even though it is not used in the update?


### Example 3 - Gradient checks and numerical scale

Finite differences are too expensive for training large models, but they are excellent sanity checks for small gradient calculations.

For one parameter vector, compare the analytical gradient with a central finite-difference approximation.


In [ ]:
def finite_difference_grad(loss_fn, theta, eps=1.0e-6):
    grad = np.zeros_like(theta)

    for j in range(theta.size):
        step = np.zeros_like(theta)
        step[j] = eps
        grad[j] = (loss_fn(theta + step) - loss_fn(theta - step)) / (2.0 * eps)

    return grad


theta0 = np.array([0.1, -0.2, 0.3])
bias0 = 0.05

loss_theta_only = lambda th: mse_loss(linear_predict(X_train_s, th, bias0), y_train)

grad_fd = finite_difference_grad(loss_theta_only, theta0)
grad_an, grad_b = linear_gradients(X_train_s, y_train, theta0, bias0)

rel_error = np.linalg.norm(grad_fd - grad_an) / np.linalg.norm(grad_fd + grad_an)

print("finite-difference gradient:", grad_fd)
print("analytical gradient:       ", grad_an)
print(f"relative difference: {rel_error:.3e}")


Now use the same learning rate on a well-scaled and an artificially badly-scaled representation of the same data. The mathematical information is similar, but the numerical optimization problem is not.


In [ ]:
def train_for_scale_demo(X_train, y_train, X_val, y_val, eta, n_steps=80):
    theta = np.zeros(X_train.shape[1])
    bias = 0.0
    history = []

    for step in range(n_steps):
        y_pred = linear_predict(X_train, theta, bias)
        loss = mse_loss(y_pred, y_train)
        history.append(loss)

        if (not np.isfinite(loss)) or loss > 1.0e80:
            break

        grad_theta, grad_bias = linear_gradients(X_train, y_train, theta, bias)
        theta -= eta * grad_theta
        bias -= eta * grad_bias

    return np.array(history)


X_train_bad = X_train_s.copy()
X_val_bad = X_val_s.copy()
X_train_bad[:, 0] *= 1.0e3
X_val_bad[:, 0] *= 1.0e3

eta = 0.05
hist_scaled = train_for_scale_demo(X_train_s, y_train, X_val_s, y_val, eta=eta)
hist_bad = train_for_scale_demo(X_train_bad, y_train, X_val_bad, y_val, eta=eta)

print("scaled representation:")
print("  steps recorded:", hist_scaled.size)
print("  last loss:     ", hist_scaled[-1])
print("  all finite:    ", np.isfinite(hist_scaled).all())

print("badly scaled representation:")
print("  steps recorded:", hist_bad.size)
print("  last loss:     ", hist_bad[-1])
print("  all finite:    ", np.isfinite(hist_bad).all())


In [ ]:
fig, ax = plt.subplots(figsize=(6.0, 3.3))

ax.plot(hist_scaled, label="scaled features")
ax.plot(hist_bad, label="one feature multiplied by 1000")

ax.set_xlabel("step")
ax.set_ylabel("loss")
ax.set_yscale("log")
ax.set_title(fr"Same learning rate, $\eta={eta}$")
ax.grid(True, alpha=0.3)
ax.legend()

fig.tight_layout()


Questions:

1. Why can the same learning rate be reasonable in one representation and unstable in another?
2. Which diagnostic first reveals the unstable run?
3. How is this similar to choosing a time step in a numerical ODE solver?


### Example 4 - Equation residuals as scientific diagnostics

A small data loss is not the only possible criterion. In scientific machine learning, we may also ask whether a model respects a known equation.

For the function

$$
u(x)=\sin(\pi x),
$$

the equation

$$
u_{xx}+\pi^2u=0
$$

is satisfied exactly. A visually small perturbation can produce a much larger equation residual.


In [ ]:
x_eq = np.linspace(0.0, 1.0, 500)

good = np.sin(np.pi * x_eq)
perturbed = good + 0.03 * np.sin(8.0 * np.pi * x_eq)


def helmholtz_residual_1d(x, u):
    dx = x[1] - x[0]
    uxx = (u[2:] - 2.0 * u[1:-1] + u[:-2]) / dx**2
    residual = uxx + np.pi**2 * u[1:-1]
    return x[1:-1], residual


xi, R_good = helmholtz_residual_1d(x_eq, good)
_, R_perturbed = helmholtz_residual_1d(x_eq, perturbed)

field_mismatch = np.sqrt(np.mean((perturbed - good)**2))
res_good = np.sqrt(np.mean(R_good**2))
res_perturbed = np.sqrt(np.mean(R_perturbed**2))

print(f"RMS field mismatch:       {field_mismatch:.3e}")
print(f"RMS equation residual, good field:      {res_good:.3e}")
print(f"RMS equation residual, perturbed field: {res_perturbed:.3e}")


In [ ]:
fig, axes = plt.subplots(2, 1, sharex=False, figsize=(6.0, 8.4))

axes[0].plot(x_eq, good, label=r"$\sin(\pi x)$")
axes[0].plot(x_eq, perturbed, "--", label="small perturbation")
axes[0].set_xlabel(r"$x$")
axes[0].set_ylabel(r"$u(x)$")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].axhline(0.0, linewidth=1)
axes[1].plot(xi, R_good, label="residual: good")
axes[1].plot(xi, R_perturbed, label="residual: perturbed")
axes[1].set_xlabel(r"$x$")
axes[1].set_ylabel(r"$u_{xx}+\pi^2u$")
axes[1].set_yscale("symlog", linthresh=1.0e-6)
axes[1].legend()
axes[1].grid(True, alpha=0.3)

fig.tight_layout()


Questions:

1. Why can two curves look close while one violates the equation much more strongly?
2. Which diagnostic would be hidden if we only plotted the model output?
3. How could an equation residual be added to a training loss?


## Part II - Independent work


### Task 1 - Honest preprocessing, baseline, and conditioning

Use the following data-generation cell:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(2026)

N = 800

length = rng.uniform(0.0, 2.0, size=N)
energy = rng.uniform(1.0e3, 5.0e3, size=N)
angle = rng.uniform(-1.0, 1.0, size=N)

X = np.column_stack([length, energy, angle])

theta_true = np.array([2.0, -1.5e-3, 0.8])
bias_true = 0.4
noise = 0.12 * rng.normal(size=N)

y = X @ theta_true + bias_true + noise

Your tasks are:

1. State the meaning of axis 0 and axis 1 of `X`.
2. Split the data into training, validation, and test sets using a fixed random permutation. Use approximately 70%, 15%, and 15% of the samples.
3. Compute the feature mean and standard deviation from the training set only.
4. Use these quantities to standardize the training, validation, and test sets.
5. Add array contracts checking shapes, finite values, and that the standardized training features have approximately zero mean and unit standard deviation.
6. Compute the training and validation loss of a mean-prediction baseline.
7. Fit a linear model using `np.linalg.lstsq` on the standardized training data. Include the bias by adding a column of ones.
8. Compute training, validation, and test losses for the fitted model.
9. Compute the condition number of

   $$
   H=\frac{1}{N}X^TX
   $$

before and after standardization. Use the training data.  
10. In two or three sentences, explain why scaling is not only cosmetic in this problem.

**Optional extension**: Repeat the standardization incorrectly by using all data before the split. Explain why this creates data leakage even if the final numbers change only slightly.


In [ ]:
#####################
# Your solution here
#####################

# ------------------------------------------------------------
# Small helper functions
# ------------------------------------------------------------

def mse_loss(y_pred, y_true):
    residual = y_pred - y_true
    return 0.5 * np.mean(residual**2)


def add_bias_column(X):
    return np.column_stack([
        X,
        np.ones(X.shape[0]),
    ])


# ------------------------------------------------------------
# Point 2
# Train / validation / test split
# ------------------------------------------------------------

split_rng = np.random.default_rng(12345)

indices = split_rng.permutation(N)

n_train = int(0.70 * N)
n_val = int(0.15 * N)

train_idx = indices[:n_train]
val_idx = indices[n_train:n_train + n_val]
test_idx = indices[n_train + n_val:]

X_train = X[train_idx]
y_train = y[train_idx]

X_val = X[val_idx]
y_val = y[val_idx]

X_test = X[test_idx]
y_test = y[test_idx]


# ------------------------------------------------------------
# Basic array contracts before preprocessing
# ------------------------------------------------------------

n_features = X.shape[1]

assert X.ndim == 2, f"X must be two-dimensional, but got X.ndim = {X.ndim}"
assert y.ndim == 1, f"y must be one-dimensional, but got y.ndim = {y.ndim}"

assert X.shape == (N, n_features)
assert y.shape == (N,)

assert X_train.shape == (n_train, n_features)
assert X_val.shape == (n_val, n_features)
assert X_test.shape == (N - n_train - n_val, n_features)

assert y_train.shape == (n_train,)
assert y_val.shape == (n_val,)
assert y_test.shape == (N - n_train - n_val,)

assert np.isfinite(X).all()
assert np.isfinite(y).all()


# ------------------------------------------------------------
# Point 3
# Compute preprocessing statistics from the training set only
# ------------------------------------------------------------

feature_mean = X_train.mean(axis=0)
feature_std = X_train.std(axis=0)

# Avoid division by zero for constant features.
feature_std_safe = feature_std.copy()
feature_std_safe[feature_std_safe == 0.0] = 1.0

assert feature_mean.shape == (n_features,)
assert feature_std_safe.shape == (n_features,)
assert np.all(feature_std_safe > 0.0)


# ------------------------------------------------------------
# Point 4
# Standardize train, validation, and test sets
# using training statistics only
# ------------------------------------------------------------

X_train_s = (X_train - feature_mean) / feature_std_safe
X_val_s = (X_val - feature_mean) / feature_std_safe
X_test_s = (X_test - feature_mean) / feature_std_safe


# ------------------------------------------------------------
# Point 5
# Contracts after standardization
# ------------------------------------------------------------

assert X_train_s.shape == X_train.shape
assert X_val_s.shape == X_val.shape
assert X_test_s.shape == X_test.shape

assert np.isfinite(X_train_s).all()
assert np.isfinite(X_val_s).all()
assert np.isfinite(X_test_s).all()

train_mean_after_scaling = X_train_s.mean(axis=0)
train_std_after_scaling = X_train_s.std(axis=0)

assert np.allclose(train_mean_after_scaling, 0.0)
assert np.allclose(train_std_after_scaling, 1.0)

print("Train shape:      ", X_train.shape)
print("Validation shape: ", X_val.shape)
print("Test shape:       ", X_test.shape)

print()
print("Training feature mean after scaling:")
print(train_mean_after_scaling)

print()
print("Training feature std after scaling:")
print(train_std_after_scaling)


# ------------------------------------------------------------
# Point 6
# Mean-prediction baseline
# ------------------------------------------------------------

baseline_value = y_train.mean()

y_train_baseline = baseline_value * np.ones_like(y_train)
y_val_baseline = baseline_value * np.ones_like(y_val)

baseline_train_loss = mse_loss(y_train_baseline, y_train)
baseline_val_loss = mse_loss(y_val_baseline, y_val)

print()
print("Baseline losses:")
print(f"training loss:   {baseline_train_loss:.8g}")
print(f"validation loss: {baseline_val_loss:.8g}")


# ------------------------------------------------------------
# Point 7
# Fit a linear model using least squares
# Include the bias by adding a column of ones
# ------------------------------------------------------------

X_train_aug = add_bias_column(X_train_s)

assert X_train_aug.shape == (n_train, n_features + 1)

coef, residuals_lstsq, rank, singular_values = np.linalg.lstsq(
    X_train_aug,
    y_train,
    rcond=None,
)

theta_fit = coef[:-1]
bias_fit = coef[-1]

assert theta_fit.shape == (n_features,)
assert np.isscalar(bias_fit)

print()
print("Fitted parameters in standardized feature space:")
print("theta:", theta_fit)
print("bias: ", bias_fit)
print("rank: ", rank)


# ------------------------------------------------------------
# Point 8
# Training, validation, and test losses for the fitted model
# ------------------------------------------------------------

X_val_aug = add_bias_column(X_val_s)
X_test_aug = add_bias_column(X_test_s)

y_train_pred = X_train_aug @ coef
y_val_pred = X_val_aug @ coef
y_test_pred = X_test_aug @ coef

assert y_train_pred.shape == y_train.shape
assert y_val_pred.shape == y_val.shape
assert y_test_pred.shape == y_test.shape

assert np.isfinite(y_train_pred).all()
assert np.isfinite(y_val_pred).all()
assert np.isfinite(y_test_pred).all()

fit_train_loss = mse_loss(y_train_pred, y_train)
fit_val_loss = mse_loss(y_val_pred, y_val)
fit_test_loss = mse_loss(y_test_pred, y_test)

print()
print("Fitted model losses:")
print(f"training loss:   {fit_train_loss:.8g}")
print(f"validation loss: {fit_val_loss:.8g}")
print(f"test loss:       {fit_test_loss:.8g}")


# ------------------------------------------------------------
# Point 9
# Condition number before and after standardization
# Use the training data
# ------------------------------------------------------------

N_train = X_train.shape[0]

H_raw = X_train.T @ X_train / N_train
H_scaled = X_train_s.T @ X_train_s / N_train

assert H_raw.shape == (n_features, n_features)
assert H_scaled.shape == (n_features, n_features)

cond_raw = np.linalg.cond(H_raw)
cond_scaled = np.linalg.cond(H_scaled)

print()
print("Condition numbers:")
print(f"before standardization: {cond_raw:.8e}")
print(f"after standardization:  {cond_scaled:.8e}")

In [ ]:
#####################
# Optional extension
#####################

# ------------------------------------------------------------
# Incorrect preprocessing: using all data before the split
# ------------------------------------------------------------

feature_mean_all = X.mean(axis=0)
feature_std_all = X.std(axis=0)

feature_std_all_safe = feature_std_all.copy()
feature_std_all_safe[feature_std_all_safe == 0.0] = 1.0

X_all_s_wrong = (X - feature_mean_all) / feature_std_all_safe

X_train_s_wrong = X_all_s_wrong[train_idx]
X_val_s_wrong = X_all_s_wrong[val_idx]
X_test_s_wrong = X_all_s_wrong[test_idx]

X_train_aug_wrong = add_bias_column(X_train_s_wrong)
X_val_aug_wrong = add_bias_column(X_val_s_wrong)
X_test_aug_wrong = add_bias_column(X_test_s_wrong)

coef_wrong, *_ = np.linalg.lstsq(
    X_train_aug_wrong,
    y_train,
    rcond=None,
)

y_train_pred_wrong = X_train_aug_wrong @ coef_wrong
y_val_pred_wrong = X_val_aug_wrong @ coef_wrong
y_test_pred_wrong = X_test_aug_wrong @ coef_wrong

wrong_train_loss = mse_loss(y_train_pred_wrong, y_train)
wrong_val_loss = mse_loss(y_val_pred_wrong, y_val)
wrong_test_loss = mse_loss(y_test_pred_wrong, y_test)

leakage_summary = {
    "wrong_train_loss": wrong_train_loss,
    "wrong_validation_loss": wrong_val_loss,
    "wrong_test_loss": wrong_test_loss,
}

leakage_summary

Scaling here is not merely a cosmetic change of units. The energy feature takes values on the order of thousands, whereas length and angle are on the order of one. Therefore, without standardization, the curvature matrix of the least-squares problem may be ill-conditioned. This affects numerical stability and would be particularly important for iterative methods, where a single learning rate would have to handle parameter directions with very different scales.

In the optional part, computing the standardization parameters on the entire matrix X is methodologically incorrect, because the means and standard deviations then contain information from the validation and test sets. Even if the final losses change only marginally, such a procedure violates the principle that validation and test data should not influence any component of model fitting or preprocessing.

### Task 2 - Gradient check and learning-rate diagnosis

Use the following setup:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(77)

N = 350

x1 = rng.normal(size=N)
x2 = 0.5 * rng.normal(size=N)
X = np.column_stack([x1, x2])

theta_true = np.array([1.5, -2.0])
bias_true = -0.4

y = X @ theta_true + bias_true + 0.15 * rng.normal(size=N)

# The features are already on a reasonable scale for this task.
print("X.shape:", X.shape)
print("y.shape:", y.shape)


Your tasks are:

1. Write a function `predict(X, theta, bias)`.
2. Write a function `loss(X, y, theta, bias)` using the factor `0.5` in front of the mean squared residual.
3. Write a function `gradients(X, y, theta, bias)` returning `grad_theta` and `grad_bias`.
4. Check that `grad_theta.shape == theta.shape`.
5. Implement a central finite-difference gradient check for `theta` at a nonzero test point, for example:

   ```python
   theta0 = np.array([0.2, -0.3])
   bias0 = 0.1
   ```

6. Report the relative difference between the finite-difference gradient and the analytical gradient.
7. Implement gradient descent and run it for several learning rates, for example:

   ```python
   eta_values = [0.001, 0.03, 0.3, 1.0]
   ```

8. Plot the loss histories on a logarithmic scale.
9. Identify which learning rates are stable, slow, or divergent.
10. Add explicit checks for non-finite losses and gradients.

In [ ]:
#####################
# Your solution here
#####################

# ------------------------------------------------------------
# Basic array contracts
# ------------------------------------------------------------

assert X.ndim == 2, f"X must be two-dimensional, but got X.ndim = {X.ndim}"
assert y.ndim == 1, f"y must be one-dimensional, but got y.ndim = {y.ndim}"

n_samples, n_features = X.shape

assert y.shape == (n_samples,)
assert np.isfinite(X).all()
assert np.isfinite(y).all()

print("n_samples:  ", n_samples)
print("n_features: ", n_features)


# ------------------------------------------------------------
# Point 1
# Prediction function
# ------------------------------------------------------------

def predict(X, theta, bias):
    if X.ndim != 2:
        raise ValueError(
            f"X must be two-dimensional, but got X.ndim = {X.ndim}"
        )

    if theta.ndim != 1:
        raise ValueError(
            f"theta must be one-dimensional, but got theta.ndim = {theta.ndim}"
        )

    if X.shape[1] != theta.size:
        raise ValueError(
            f"X has {X.shape[1]} features, but theta has size {theta.size}"
        )

    return X @ theta + bias


# ------------------------------------------------------------
# Point 2
# Loss function with factor 0.5
# ------------------------------------------------------------

def loss(X, y, theta, bias):
    y_pred = predict(X, theta, bias)

    if y_pred.shape != y.shape:
        raise ValueError(
            f"prediction shape {y_pred.shape} does not match y shape {y.shape}"
        )

    residual = y_pred - y
    return 0.5 * np.mean(residual**2)


# ------------------------------------------------------------
# Point 3
# Analytical gradients
# ------------------------------------------------------------

def gradients(X, y, theta, bias):
    y_pred = predict(X, theta, bias)
    residual = y_pred - y

    N = residual.size

    grad_theta = X.T @ residual / N
    grad_bias = residual.mean()

    return grad_theta, grad_bias

In [ ]:
# ------------------------------------------------------------
# Point 4, 5, and 6
# Central finite-difference gradient check
# ------------------------------------------------------------

def finite_difference_grad_theta(X, y, theta, bias, eps=1.0e-6):
    grad_fd = np.zeros_like(theta)

    for j in range(theta.size):
        step = np.zeros_like(theta)
        step[j] = eps

        loss_plus = loss(X, y, theta + step, bias)
        loss_minus = loss(X, y, theta - step, bias)

        grad_fd[j] = (loss_plus - loss_minus) / (2.0 * eps)

    return grad_fd


theta0 = np.array([0.2, -0.3])
bias0 = 0.1

grad_theta, grad_bias = gradients(X, y, theta0, bias0)
grad_theta_fd = finite_difference_grad_theta(X, y, theta0, bias0)

assert grad_theta.shape == theta0.shape
assert grad_theta_fd.shape == theta0.shape

relative_difference = np.linalg.norm(grad_theta - grad_theta_fd) / (
    np.linalg.norm(grad_theta) + np.linalg.norm(grad_theta_fd)
)

print("Analytical grad_theta:")
print(grad_theta)

print()
print("Finite-difference grad_theta:")
print(grad_theta_fd)

print()
print(f"Relative difference: {relative_difference:.8e}")

assert np.allclose(grad_theta, grad_theta_fd, rtol=1.0e-5, atol=1.0e-7)

In [ ]:
# ------------------------------------------------------------
# Point 7 and 10
# Gradient descent with explicit non-finite checks
# ------------------------------------------------------------

def run_gradient_descent(X, y, eta, n_steps=300):
    theta = np.zeros(X.shape[1])
    bias = 0.0

    history = []
    status = "finished"

    for step in range(n_steps):
        current_loss = loss(X, y, theta, bias)

        if not np.isfinite(current_loss):
            status = "stopped: non-finite loss"
            break

        grad_theta, grad_bias = gradients(X, y, theta, bias)

        if not np.isfinite(grad_theta).all():
            status = "stopped: non-finite grad_theta"
            break

        if not np.isfinite(grad_bias):
            status = "stopped: non-finite grad_bias"
            break

        history.append(current_loss)

        theta = theta - eta * grad_theta
        bias = bias - eta * grad_bias

    history = np.array(history)

    return {
        "eta": eta,
        "theta": theta,
        "bias": bias,
        "history": history,
        "status": status,
    }


eta_values = [0.001, 0.03, 0.3, 1.0, 1.92]

runs = []

for eta in eta_values:
    result = run_gradient_descent(
        X,
        y,
        eta=eta,
        n_steps=300,
    )

    runs.append(result)

    print(
        f"eta = {eta:7.4f}, "
        f"steps recorded = {result['history'].size:4d}, "
        f"initial loss = {result['history'][0]:.6g}, "
        f"final loss = {result['history'][-1]:.6g}, "
        f"status = {result['status']}"
    )

In [ ]:
# ------------------------------------------------------------
# Point 8
# Plot loss histories on a logarithmic scale
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(6.2, 4.0))

for result in runs:
    eta = result["eta"]
    history = result["history"]

    if history.size > 0:
        ax.plot(
            history,
            label=fr"$\eta = {eta}$",
        )

ax.set_xlabel("gradient descent step")
ax.set_ylabel("loss")
ax.set_yscale("log")
ax.set_title("Learning-rate diagnosis from loss histories")
ax.grid(True, which="both", alpha=0.3)
ax.legend()

fig.tight_layout()
plt.show()

For the original, reasonably scaled features, eta = 0.001 is stable but slow. The values eta = 0.03, eta = 0.3, and also eta = 1.0 are stable for this particular dataset, with the larger stable values converging much faster. In this specific list there are no truly divergent case; trying a still larger value, for example around eta = 1.92, would reveal divergent behaviour.

The finite-difference gradient and the analytical gradient should agree to a very small relative difference, usually around machine precision for this problem. This check is useful because it verifies the gradient formula independently of the training loop.

### Task 3 - A small physics-informed learning workflow

We observe noisy samples of a function on the interval $[0,1]$. The underlying clean function approximately satisfies the Helmholtz equation

$$
u_{xx}+\pi^2u=0.
$$

We will fit the observations with a sine basis

$$
u_\theta(x)=\sum_{m=1}^{M}\theta_m\sin(m\pi x).
$$

The model is linear in the parameters, but the workflow has the same structure as a small machine-learning experiment: data, model, loss, gradient, validation, and residual diagnostics.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(2026)

n_data = 90
x_data = np.sort(rng.uniform(0.0, 1.0, size=n_data))

u_clean = np.sin(np.pi * x_data)
sigma = 0.04 * (1.0 + 0.5 * x_data)
u_obs = u_clean + sigma * rng.normal(size=n_data)

idx = rng.permutation(n_data)
n_train = int(0.65 * n_data)
n_val = int(0.20 * n_data)

train_idx = idx[:n_train]
val_idx = idx[n_train:n_train + n_val]
test_idx = idx[n_train + n_val:]

modes = np.arange(1, 9)


def basis(x, modes):
    return np.sin(np.pi * x[:, None] * modes[None, :])


def residual_basis(x, modes):
    # Basis for the normalized residual (u_xx + pi^2 u) / pi^2.
    Phi = basis(x, modes)
    factors = 1.0 - modes**2
    return Phi * factors[None, :]


x_collocation = np.linspace(0.0, 1.0, 300)
x_plot = np.linspace(0.0, 1.0, 500)
u_true_plot = np.sin(np.pi * x_plot)

print("number of modes:", modes.size)
print("training samples:", train_idx.size)
print("validation samples:", val_idx.size)
print("test samples:", test_idx.size)


Your tasks are:

1. Construct the basis matrices for the training, validation, test, collocation, and plotting points.
2. Write a function `predict(Phi, theta)` returning `Phi @ theta`.
3. Define a weighted data loss on a subset:

   $$
   L_{\mathrm{data}}=\frac{1}{2N}\sum_i\left(\frac{u_\theta(x_i)-u_i}{\sigma_i}\right)^2.
   $$

4. Define a normalized equation-residual loss on the collocation grid:

   $$
   L_{\mathrm{pde}}=\frac{1}{2N_c}\sum_j\left(\frac{u_{\theta,xx}(x_j)+\pi^2u_\theta(x_j)}{\pi^2}\right)^2.
   $$

   The helper function `residual_basis` already gives the basis for the normalized residual.
5. Define the combined loss

   $$
   L = L_{\mathrm{data}} + \alpha L_{\mathrm{pde}} + \frac{\lambda}{2}\|\theta\|^2.
   $$

6. Derive and implement the gradient of this combined loss with respect to `theta`.
7. Train the model with gradient descent for at least three values of `alpha`, for example:

   ```python
   alpha_values = [0.0, 0.1, 1.0]
   lambda_reg = 1.0e-5
   eta = 5.0e-4
   n_steps = 3000
   ```

    If a run becomes non-finite, reduce the learning rate and record that this was necessary.  
8. During training, record:
   - total training loss;
   - training data loss;
   - validation data loss;
   - PDE residual loss;
   - gradient norm.
9. Plot the training and validation data-loss histories for each value of `alpha`.
10. Plot the noisy data with error bars and overlay the fitted functions on a dense grid.
11. For each value of `alpha`, compute and report:
    - final training data loss;
    - validation data loss;
    - test data loss;
    - PDE residual loss;
    - RMS test error in physical units, not divided by `sigma`.
12. Plot the equation residual as a function of `x` for the fitted models.

In [ ]:
#####################
# Your solution here
#####################

# ------------------------------------------------------------
# Basic array contracts
# ------------------------------------------------------------

assert x_data.ndim == 1
assert u_obs.ndim == 1
assert sigma.ndim == 1

assert x_data.shape == u_obs.shape
assert sigma.shape == u_obs.shape

assert np.isfinite(x_data).all()
assert np.isfinite(u_obs).all()
assert np.isfinite(sigma).all()

assert np.all(sigma > 0.0)

assert modes.ndim == 1
assert np.all(modes >= 1)

n_modes = modes.size

print("number of data points:", x_data.size)
print("number of modes:      ", n_modes)
print("training samples:     ", train_idx.size)
print("validation samples:   ", val_idx.size)
print("test samples:         ", test_idx.size)

In [ ]:
# ------------------------------------------------------------
# Point 1
# Construct basis matrices
# ------------------------------------------------------------

x_train = x_data[train_idx]
u_train = u_obs[train_idx]
sigma_train = sigma[train_idx]

x_val = x_data[val_idx]
u_val = u_obs[val_idx]
sigma_val = sigma[val_idx]

x_test = x_data[test_idx]
u_test = u_obs[test_idx]
sigma_test = sigma[test_idx]

Phi_train = basis(x_train, modes)
Phi_val = basis(x_val, modes)
Phi_test = basis(x_test, modes)

Phi_collocation = basis(x_collocation, modes)
R_collocation = residual_basis(x_collocation, modes)

Phi_plot = basis(x_plot, modes)
R_plot = residual_basis(x_plot, modes)

assert Phi_train.shape == (train_idx.size, n_modes)
assert Phi_val.shape == (val_idx.size, n_modes)
assert Phi_test.shape == (test_idx.size, n_modes)

assert Phi_collocation.shape == (x_collocation.size, n_modes)
assert R_collocation.shape == (x_collocation.size, n_modes)

assert Phi_plot.shape == (x_plot.size, n_modes)
assert R_plot.shape == (x_plot.size, n_modes)

assert np.isfinite(Phi_train).all()
assert np.isfinite(Phi_val).all()
assert np.isfinite(Phi_test).all()
assert np.isfinite(R_collocation).all()

print("Phi_train shape:       ", Phi_train.shape)
print("Phi_val shape:         ", Phi_val.shape)
print("Phi_test shape:        ", Phi_test.shape)
print("R_collocation shape:   ", R_collocation.shape)
print("Phi_plot shape:        ", Phi_plot.shape)

In [ ]:
# ------------------------------------------------------------
# Point 2
# Prediction function
# ------------------------------------------------------------

def predict(Phi, theta):
    if Phi.ndim != 2:
        raise ValueError(
            f"Phi must be two-dimensional, but got Phi.ndim = {Phi.ndim}"
        )

    if theta.ndim != 1:
        raise ValueError(
            f"theta must be one-dimensional, but got theta.ndim = {theta.ndim}"
        )

    if Phi.shape[1] != theta.size:
        raise ValueError(
            f"Phi has {Phi.shape[1]} columns, but theta has size {theta.size}"
        )

    return Phi @ theta


# ------------------------------------------------------------
# Point 3
# Weighted data loss
# ------------------------------------------------------------

def data_loss(Phi, u, sigma, theta):
    u_pred = predict(Phi, theta)

    if u_pred.shape != u.shape:
        raise ValueError(
            f"prediction shape {u_pred.shape} does not match u shape {u.shape}"
        )

    if sigma.shape != u.shape:
        raise ValueError(
            f"sigma shape {sigma.shape} does not match u shape {u.shape}"
        )

    weighted_residual = (u_pred - u) / sigma

    return 0.5 * np.mean(weighted_residual**2)


# ------------------------------------------------------------
# Point 4
# Normalized PDE residual loss
# ------------------------------------------------------------

def pde_loss(R, theta):
    residual = predict(R, theta)
    return 0.5 * np.mean(residual**2)


# ------------------------------------------------------------
# Point 5
# Combined loss
# ------------------------------------------------------------

def combined_loss(
    Phi_data,
    u_data,
    sigma_data,
    R_collocation,
    theta,
    alpha,
    lambda_reg,
):
    L_data = data_loss(Phi_data, u_data, sigma_data, theta)
    L_pde = pde_loss(R_collocation, theta)
    L_reg = 0.5 * lambda_reg * np.sum(theta**2)

    return L_data + alpha * L_pde + L_reg

# ------------------------------------------------------------
# Point 6
# Gradient of the combined loss
# ------------------------------------------------------------

def data_gradient(Phi, u, sigma, theta):
    u_pred = predict(Phi, theta)

    # L_data = 1/(2N) sum_i ((Phi theta - u_i) / sigma_i)^2
    #
    # dL_data/dtheta =
    # 1/N Phi.T @ ((Phi theta - u) / sigma^2)

    residual = u_pred - u
    grad = Phi.T @ (residual / sigma**2) / u.size

    return grad


def pde_gradient(R, theta):
    residual = predict(R, theta)

    # L_pde = 1/(2Nc) sum_j (R theta)_j^2
    #
    # dL_pde/dtheta = 1/Nc R.T @ (R theta)

    grad = R.T @ residual / residual.size

    return grad


def combined_gradient(
    Phi_data,
    u_data,
    sigma_data,
    R_collocation,
    theta,
    alpha,
    lambda_reg,
):
    grad_data = data_gradient(Phi_data, u_data, sigma_data, theta)
    grad_pde = pde_gradient(R_collocation, theta)
    grad_reg = lambda_reg * theta

    grad_total = grad_data + alpha * grad_pde + grad_reg

    return grad_total

In [ ]:
# ------------------------------------------------------------
# Optional gradient shape and finite-value check at a test point
# ------------------------------------------------------------

theta0 = np.zeros(n_modes)
theta0[0] = 0.5
theta0[1] = -0.2

alpha0 = 0.1
lambda_reg = 1.0e-5

test_loss = combined_loss(
    Phi_train,
    u_train,
    sigma_train,
    R_collocation,
    theta0,
    alpha0,
    lambda_reg,
)

test_grad = combined_gradient(
    Phi_train,
    u_train,
    sigma_train,
    R_collocation,
    theta0,
    alpha0,
    lambda_reg,
)

assert np.isfinite(test_loss)
assert test_grad.shape == theta0.shape
assert np.isfinite(test_grad).all()

print("test combined loss:", test_loss)
print("test gradient norm:", np.linalg.norm(test_grad))

In [ ]:
# ------------------------------------------------------------
# Point 7 and 8
# Gradient descent training
# ------------------------------------------------------------

def train_model(
    Phi_train,
    u_train,
    sigma_train,
    Phi_val,
    u_val,
    sigma_val,
    R_collocation,
    alpha,
    lambda_reg,
    eta,
    n_steps,
):
    theta = np.zeros(Phi_train.shape[1])

    history = {
        "total_train_loss": [],
        "train_data_loss": [],
        "val_data_loss": [],
        "pde_loss": [],
        "grad_norm": [],
    }

    status = "finished"

    for step in range(n_steps):
        L_train_data = data_loss(
            Phi_train,
            u_train,
            sigma_train,
            theta,
        )

        L_val_data = data_loss(
            Phi_val,
            u_val,
            sigma_val,
            theta,
        )

        L_pde = pde_loss(
            R_collocation,
            theta,
        )

        L_total = (
            L_train_data
            + alpha * L_pde
            + 0.5 * lambda_reg * np.sum(theta**2)
        )

        grad = combined_gradient(
            Phi_train,
            u_train,
            sigma_train,
            R_collocation,
            theta,
            alpha,
            lambda_reg,
        )

        grad_norm = np.linalg.norm(grad)

        # Explicit non-finite checks.
        if not np.isfinite(L_total):
            status = "stopped: non-finite total loss"
            break

        if not np.isfinite(L_train_data):
            status = "stopped: non-finite training data loss"
            break

        if not np.isfinite(L_val_data):
            status = "stopped: non-finite validation data loss"
            break

        if not np.isfinite(L_pde):
            status = "stopped: non-finite PDE loss"
            break

        if not np.isfinite(grad).all():
            status = "stopped: non-finite gradient"
            break

        history["total_train_loss"].append(L_total)
        history["train_data_loss"].append(L_train_data)
        history["val_data_loss"].append(L_val_data)
        history["pde_loss"].append(L_pde)
        history["grad_norm"].append(grad_norm)

        theta = theta - eta * grad

    for name in history:
        history[name] = np.array(history[name])

    return {
        "alpha": alpha,
        "lambda_reg": lambda_reg,
        "eta": eta,
        "theta": theta,
        "history": history,
        "status": status,
    }


alpha_values = [0.0, 0.1, 1.0]
lambda_reg = 1.0e-5
eta = 5.0e-4
n_steps = 3000

runs = []

for alpha in alpha_values:
    result = train_model(
        Phi_train=Phi_train,
        u_train=u_train,
        sigma_train=sigma_train,
        Phi_val=Phi_val,
        u_val=u_val,
        sigma_val=sigma_val,
        R_collocation=R_collocation,
        alpha=alpha,
        lambda_reg=lambda_reg,
        eta=eta,
        n_steps=n_steps,
    )

    # If something becomes non-finite, retry once with a smaller learning rate.
    if result["status"] != "finished":
        eta_reduced = 0.2 * eta

        result = train_model(
            Phi_train=Phi_train,
            u_train=u_train,
            sigma_train=sigma_train,
            Phi_val=Phi_val,
            u_val=u_val,
            sigma_val=sigma_val,
            R_collocation=R_collocation,
            alpha=alpha,
            lambda_reg=lambda_reg,
            eta=eta_reduced,
            n_steps=n_steps,
        )

        result["eta_was_reduced"] = True
    else:
        result["eta_was_reduced"] = False

    runs.append(result)

    h = result["history"]

    print(
        f"alpha = {alpha:4.1f}, "
        f"eta = {result['eta']:.2e}, "
        f"steps = {h['total_train_loss'].size:4d}, "
        f"final train data loss = {h['train_data_loss'][-1]:.6g}, "
        f"final val data loss = {h['val_data_loss'][-1]:.6g}, "
        f"final PDE loss = {h['pde_loss'][-1]:.6g}, "
        f"status = {result['status']}, "
        f"eta reduced = {result['eta_was_reduced']}"
    )

In [ ]:
# ------------------------------------------------------------
# Point 9
# Plot training and validation data-loss histories
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(6.6, 4.2))

for result in runs:
    alpha = result["alpha"]
    h = result["history"]

    ax.plot(
        h["train_data_loss"],
        label=fr"train, $\alpha={alpha}$",
    )

    ax.plot(
        h["val_data_loss"],
        linestyle="--",
        label=fr"validation, $\alpha={alpha}$",
    )

ax.set_xlabel("gradient descent step")
ax.set_ylabel("weighted data loss")
ax.set_yscale("log")
ax.set_title("Training and validation data-loss histories")
ax.grid(True, which="both", alpha=0.3)
ax.legend()

fig.tight_layout()
plt.show()

In [ ]:
# ------------------------------------------------------------
# Point 10
# Plot noisy data with error bars and fitted functions
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(7.0, 4.2))

ax.errorbar(
    x_data,
    u_obs,
    yerr=sigma,
    fmt=".",
    markersize=4,
    capsize=2,
    label="noisy observations",
)

ax.plot(
    x_plot,
    u_true_plot,
    linewidth=2,
    label=r"clean reference $\sin(\pi x)$",
)

for result in runs:
    alpha = result["alpha"]
    theta = result["theta"]

    u_fit_plot = predict(Phi_plot, theta)

    ax.plot(
        x_plot,
        u_fit_plot,
        label=fr"fit, $\alpha={alpha}$",
    )

ax.set_xlabel(r"$x$")
ax.set_ylabel(r"$u(x)$")
ax.set_title("Physics-informed sine-basis fits")
ax.grid(True, alpha=0.3)
ax.legend()

fig.tight_layout()
plt.show()

In [ ]:
# ------------------------------------------------------------
# Point 11
# Final diagnostics for each alpha
# ------------------------------------------------------------

summary = []

for result in runs:
    alpha = result["alpha"]
    theta = result["theta"]

    final_train_data_loss = data_loss(
        Phi_train,
        u_train,
        sigma_train,
        theta,
    )

    final_val_data_loss = data_loss(
        Phi_val,
        u_val,
        sigma_val,
        theta,
    )

    final_test_data_loss = data_loss(
        Phi_test,
        u_test,
        sigma_test,
        theta,
    )

    final_pde_loss = pde_loss(
        R_collocation,
        theta,
    )

    u_test_pred = predict(Phi_test, theta)
    rms_test_error = np.sqrt(np.mean((u_test_pred - u_test)**2))

    summary.append({
        "alpha": alpha,
        "train_data_loss": final_train_data_loss,
        "val_data_loss": final_val_data_loss,
        "test_data_loss": final_test_data_loss,
        "pde_loss": final_pde_loss,
        "rms_test_error": rms_test_error,
    })


print("Final diagnostics:")
print(
    f"{'alpha':>8s} "
    f"{'train loss':>14s} "
    f"{'val loss':>14s} "
    f"{'test loss':>14s} "
    f"{'PDE loss':>14s} "
    f"{'RMS test error':>16s}"
)

for row in summary:
    print(
        f"{row['alpha']:8.3g} "
        f"{row['train_data_loss']:14.6g} "
        f"{row['val_data_loss']:14.6g} "
        f"{row['test_data_loss']:14.6g} "
        f"{row['pde_loss']:14.6g} "
        f"{row['rms_test_error']:16.6g}"
    )

In [ ]:
# ------------------------------------------------------------
# Point 12
# Plot the normalized equation residual as a function of x
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(7.0, 4.2))

ax.axhline(
    0.0,
    linewidth=1,
)

for result in runs:
    alpha = result["alpha"]
    theta = result["theta"]

    residual_plot = predict(R_plot, theta)

    ax.plot(
        x_plot,
        residual_plot,
        label=fr"$\alpha={alpha}$",
    )

ax.set_xlabel(r"$x$")
ax.set_ylabel(r"normalized residual $(u_{xx}+\pi^2u)/\pi^2$")
ax.set_title("Equation residual for the fitted models")
ax.grid(True, alpha=0.3)
ax.legend()

fig.tight_layout()
plt.show()

A useful way to read the result is that alpha = 0 fits only the noisy observations, while larger alpha also penalizes components that violate the Helmholtz equation. Since the mode m = 1 has zero normalized residual factor, the PDE term encourages the fit to prefer the physically compatible first sine mode and suppress higher modes.

The validation and test losses tell us whether this physics-informed penalty improves generalization or merely makes the training loss larger. The residual plot is especially important here: two fitted curves may look similar in data space, but their equation residuals can differ strongly.